In [5]:
import pandas as pd
import os
import numpy as np
from scipy import stats
from tqdm import tqdm


noises = [0] + [round(i*0.01, 2) for i in range(1, 11)]
tests = {
    "test": "roc_aucs_test",
    "testAB": "roc_aucs_testAB",
    "testAG": "roc_aucs_testAG"
}

methods_lst = ['grad01_avg', 'grad01_max', 'gradient_conf_avg', 
               'gradient_conf_max', 'gradient_input_avg', 'gradient_input_max', 
               'model_grad_avg', 
               'model_grad_max', 'qbc', 'threshold_avg', 
               'threshold_max', 'feature_eucl', 'random']

stats_dir = "results/stats"
os.makedirs("../"+stats_dir, exist_ok=True)

def compute_auc_per_run_method(df, value_col, x_col='ags_number', run_col='run_id', method_col='method'):
    records = []
    for (run_id, method), sub in df.groupby([run_col, method_col]):
        sub2 = sub.dropna(subset=[x_col, value_col])
        if sub2.empty:
            auc = np.nan
        else:
            s = sub2.groupby(x_col)[value_col].mean().sort_index()
            if s.shape[0] >= 2:
                x = s.index.astype(float).to_numpy()
                y = s.values.astype(float)
                auc = np.trapz(y=y, x=x)
            else:
                auc = np.nan
        records.append({'run_id': str(run_id), 'method': method, 'auc': float(auc) if not np.isnan(auc) else np.nan})
    return pd.DataFrame.from_records(records)

def paired_tests_vs_random(aucs_long_df, alpha=0.05, alternative='two-sided'):
    results = []
    tests_list = sorted(aucs_long_df['test'].unique())
    for test in tests_list:
        sub = aucs_long_df[aucs_long_df['test'] == test]
        methods = sorted(sub['method'].unique())
        if 'random' not in methods:
            print(f"Test {test}: no 'random' method found — skipping")
            continue
        methods_to_test = [m for m in methods if m != 'random']
        m = len(methods_to_test)
        for method in methods_to_test:
            a = sub[sub['method'] == method][['run_id','auc']].set_index('run_id')
            b = sub[sub['method'] == 'random'][['run_id','auc']].set_index('run_id')
            paired = a.join(b, how='inner', lsuffix='_m', rsuffix='_r').dropna()
            n_pairs = paired.shape[0]
            if n_pairs < 2:
                results.append({
                    'test': test, 'method': method, 'n_pairs': n_pairs,
                    'mean_method': np.nan, 'mean_random': np.nan, 'mean_diff': np.nan,
                    't_stat': np.nan, 'p_val': np.nan, 'p_adj': np.nan, 'reject': False,
                    'cohen_d': np.nan, 'ci_lower_rel_pct': np.nan, 'ci_upper_rel_pct': np.nan
                })
                continue

            x = paired['auc_m'].to_numpy(dtype=float)
            y = paired['auc_r'].to_numpy(dtype=float)
            diff = x - y
            mean_method = x.mean()
            mean_random = y.mean()
            mean_diff = diff.mean()
            sd_diff = diff.std(ddof=1)

            try:
                t_res = stats.ttest_rel(x, y, alternative=alternative)
                t_stat = float(t_res.statistic)
                p_val = float(t_res.pvalue)
            except TypeError:
                # older scipy without 'alternative' argument
                t_stat, p_two = stats.ttest_rel(x, y)
                if alternative == 'two-sided':
                    p_val = float(p_two)
                elif alternative == 'greater':
                    p_val = float(p_two/2) if t_stat > 0 else 1.0 - float(p_two/2)
                elif alternative == 'less':
                    p_val = float(p_two/2) if t_stat < 0 else 1.0 - float(p_two/2)
                else:
                    p_val = float(p_two)

            cohen_d = (mean_diff / sd_diff) if sd_diff > 0 else np.nan

            if n_pairs > 1:
                se_diff = sd_diff / np.sqrt(n_pairs)
                # two-sided critical t
                t_crit = stats.t.ppf(1 - alpha/2, df=n_pairs - 1)
                ci_lower = (mean_diff - t_crit * se_diff)/mean_random*100
                ci_upper = (mean_diff + t_crit * se_diff)/mean_random*100
            else:
                ci_lower = np.nan
                ci_upper = np.nan

            # Bonferroni correction across methods for this test
            p_adj = min(p_val * m, 1.0) if m > 0 else p_val
            reject = (p_adj < alpha)

            results.append({
                'test': test, 'method': method, 'n_pairs': n_pairs,
                'mean_method': mean_method, 'mean_random': mean_random, 'mean_diff': mean_diff,
                't_stat': t_stat, 'p_val': p_val, 'p_adj': p_adj, 'reject': bool(reject),
                'cohen_d': cohen_d, 'ci_lower_rel_pct': ci_lower, 'ci_upper_rel_pct': ci_upper
            })

    results_df = pd.DataFrame(results)
    cols = ['test','method','n_pairs','mean_method','mean_random','mean_diff',
            't_stat','p_val','p_adj','reject','ci_lower_rel_pct','ci_upper_rel_pct']
    results_df = results_df[[c for c in cols if c in results_df.columns]]
    return results_df

def calculate_ttest_results(df):
    aucs_dfs = []
    for test_name, col in tests.items():
        auc = compute_auc_per_run_method(df, value_col=col, x_col='ags_number', run_col='run_id', method_col='method')
        auc['test'] = test_name
        aucs_dfs.append(auc)
    aucs_long = pd.concat(aucs_dfs, ignore_index=True)
    ttest_results = paired_tests_vs_random(aucs_long, alpha=0.05, alternative='greater')
    return ttest_results
    

In [2]:
os.makedirs("temp/", exist_ok=True)

for noise in noises:
    dfn = pd.DataFrame()
    for n in tqdm(range(500)):
        for m in methods_lst:
            filepath = f"../secrun/noise{noise}"+'/rand'+str(n)+'/roc_aucs_'+m+f'_{n}.tsv'
            try:
                dfx = pd.read_csv(filepath, sep='\t')
                dfx['noise'] = noise
                dfx['method'] = m
                dfx['run_id'] = n
                dfn = pd.concat([dfn, dfx])
            except:
                pass

    df = dfn[['roc_aucs_test', 'roc_aucs_testAB', 
              'roc_aucs_testAG', 'ags_number', 'run_id', 'noise', 
              'method']].drop_duplicates().reset_index(drop=True)
    df.to_csv(f'temp/noise_{noise}.tsv', index=None, sep='\t')


100%|█████████████████████████████████████████| 500/500 [02:37<00:00,  3.18it/s]


In [3]:
for noise in noises:
    df = pd.read_csv(f'temp/noise_{noise}.tsv', sep='\t')
    ttest_results = calculate_ttest_results(df)
    ttest_results['mean_diff_rel_pct'] = ttest_results.mean_diff/ttest_results.mean_random*100
    out_tsv = os.path.join("..", stats_dir, f"stats_noise_{noise}.tsv")
    ttest_results.to_csv(out_tsv, index=None, sep='\t')


In [7]:
import pandas as pd


noises = [0] + [round(i*0.01, 2) for i in range(1, 11)]

df = pd.DataFrame()
for noise in noises:
    dfn = pd.read_csv(f'temp/noise_{noise}.tsv', sep='\t')
    df = pd.concat([df, dfn])
ttest_results = calculate_ttest_results(df)
ttest_results['mean_diff_rel_pct'] = ttest_results.mean_diff/ttest_results.mean_random*100
out_tsv = os.path.join("..", stats_dir, f"stats.tsv")
ttest_results.to_csv(out_tsv, index=None, sep='\t')
df.to_csv('temp/all.tsv', index=None, sep='\t')